# ==================================================
# **FINAL MACHINE LEARNING & DEEP LEARNING PROJECT** 
# ==================================================

#  Multimodal Emotion AI: Human Emotion Recognition
**Project by:** Ayesha Tabassum    

---

##  Project Objective
The objective of this research project is to develop a **Multimodal Emotion Recognition System** capable of identifying human emotions by fusing features from three distinct modalities: **Facial Expressions (Computer Vision)**, **Speech Patterns (Audio Analysis)**, and **Textual Sentiment (NLP)**. By comparing classical Machine Learning models with Deep Learning architectures, this project aims to create a robust system that mimics human-like perception.

---

##  Dataset References
This project utilizes the following benchmark datasets:

1. **FER2013 (Facial Emotion Recognition):** 
  [https://www.kaggle.com/datasets/deadskull7/fer2013](https://www.kaggle.com/datasets/deadskull7/fer2013)
2. **RAVDESS (Speech Emotion Recognition):** 
   [https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio](https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio)
3. **GoEmotions (Text Emotion Recognition):** 
   [https://www.kaggle.com/datasets/debarshichanda/goemotions](https://www.kaggle.com/datasets/debarshichanda/goemotions)

---

##  Project Roadmap
*   **Module 1:** Data Analytics & Preprocessing
*   **Module 2:** Classical Machine Learning Comparison (SVM, Random Forest)
*   **Module 3:** Deep Learning & Computer Vision (CNN, Transfer Learning)
*   **Module 4:** NLP & Speech Feature Extraction
*   **Module 5:** Multimodal Fusion & Final Dashboard

----------------------------------------------------------------------
## STEP 1: IMPORT LIBRARIES AND SETUP ENVIRONMENT
----------------------------------------------------------------------

In [ ]:
# 1. Standard Data Handling & Visualization
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm 
import scipy.stats as stats
# 2. Computer Vision & Deep Learning (FER2013)
import cv2            
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 3. Audio Processing (RAVDESS)
import librosa        
import librosa.display

# 4. NLP & Text Analysis (GoEmotions)
import nltk
from nltk.corpus import stopwords
import re
from sklearn.feature_extraction.text import TfidfVectorizer

# 5. Machine Learning & Evaluation (Comparison Phase)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,  roc_curve, auc
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

plt.rcParams['figure.dpi'] = 300 
sns.set_theme(style="whitegrid")

# ---- Reproducibility Setting ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Warnings and Display Settings ----
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# ---- Plot Style Settings (Professional Look) ----
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"

print("All specialized libraries imported successfully.")
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

---
## STEP 2 (DEBUG): DEEP EXPLORE /kaggle/input/ STRUCTURE
---

In [ ]:
def explore_directory(path, max_depth=3, current_depth=0):
    if current_depth > max_depth:
        return
    
    try:
        items = os.listdir(path)
    except Exception as e:
        print("  " * current_depth + f"(Cannot read: {e})")
        return
    
    for item in items:
        full_path = os.path.join(path, item)
        indent = "  " * current_depth
        
        if os.path.isdir(full_path):
            print(f"{indent}📁 {item}")
            explore_directory(full_path, max_depth, current_depth + 1)
        else:
            print(f"{indent}📄 {item}")

print("Full structure of /kaggle/input/:\n")
explore_directory("/kaggle/input")

---------------------------------------------------------------------
## STEP 3: LOAD FER2013 DATASET & EXPLORATORY DATA ANALYSIS
---------------------------------------------------------------------

In [ ]:
# ---- Define Global Dataset Paths (we'll reuse these throughout the project) ----
FER2013_PATH   = "/kaggle/input/datasets/deadskull7/fer2013/fer2013.csv"
RAVDESS_PATH   = "/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio/audio_speech_actors_01-24"
GOEMOTIONS_DIR = "/kaggle/input/datasets/debarshichanda/goemotions/data"

# ---- Load FER2013 into a DataFrame ----
fer_df = pd.read_csv(FER2013_PATH)

# ---- Basic Structure Check ----
print("Shape of FER2013 dataset:", fer_df.shape)
print("\nColumn names:", list(fer_df.columns))
print("\nFirst 5 rows:")
display(fer_df.head())

# ---- Map numeric emotion labels to readable names ----
emotion_map = {
    0: "Angry", 1: "Disgust", 2: "Fear",
    3: "Happy", 4: "Sad", 5: "Surprise", 6: "Neutral"
}
fer_df["emotion_label"] = fer_df["emotion"].map(emotion_map)

# ---- Check class distribution (Statistics Module) ----
print("\nClass distribution (counts):")
print(fer_df["emotion_label"].value_counts())
import matplotlib.pyplot as plt
import numpy as np

# ---- Prepare data ----
emotion_counts = fer_df["emotion_label"].value_counts()
labels = emotion_counts.index.tolist()
values = emotion_counts.values
n = len(labels)

# ---- Vibrant color palette (one distinct color per emotion) ----
colors = plt.cm.rainbow(np.linspace(0, 1, n))

# ---- Create figure with 2 subplots: Polar Bar (left) + Donut (right) ----
fig = plt.figure(figsize=(16, 8))
fig.patch.set_facecolor('white')

# =========================================================
# LEFT PLOT: Circular / Polar Bar Chart
# =========================================================
ax1 = fig.add_subplot(1, 2, 1, polar=True)

angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
bar_width = (2 * np.pi / n) * 0.85

bars = ax1.bar(
    angles, values,
    width=bar_width,
    color=colors,
    edgecolor="white",
    linewidth=2,
    alpha=0.9
)

# Style the polar plot
ax1.set_theta_offset(np.pi / 2)          # start from top
ax1.set_theta_direction(-1)              # clockwise
ax1.set_xticks(angles)
ax1.set_xticklabels(labels, fontsize=11, fontweight="bold")
ax1.set_yticklabels([])                  # hide radial numbers for clean look
ax1.spines['polar'].set_visible(False)
ax1.set_facecolor("#f7f7f7")

# Add value labels at the end of each bar
for angle, value, color in zip(angles, values, colors):
    ax1.text(
        angle, value + (max(values) * 0.08), f"{value}",
        ha="center", va="center", fontsize=10, fontweight="bold", color="black"
    )

ax1.set_title("Emotion Distribution — Circular Bar Chart", 
               fontsize=14, fontweight="bold", pad=30)

# =========================================================
# RIGHT PLOT: Donut Chart
# =========================================================
ax2 = fig.add_subplot(1, 2, 2)

wedges, texts, autotexts = ax2.pie(
    values,
    labels=labels,
    colors=colors,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.82,
    wedgeprops=dict(width=0.4, edgecolor="white", linewidth=2),
    textprops=dict(fontsize=11, fontweight="bold")
)

for autotext in autotexts:
    autotext.set_color("white")
    autotext.set_fontsize(9)

# Center text showing total images
ax2.text(0, 0, f"Total\n{sum(values):,}\nImages",
          ha="center", va="center", fontsize=13, fontweight="bold")

ax2.set_title("Emotion Distribution — Donut Chart", 
               fontsize=14, fontweight="bold", pad=20)

plt.suptitle("FER2013 Dataset — Emotion Class Distribution", 
              fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## STEP 4: SAMPLE IMAGE GALLERY — FER2013
---

In [ ]:
def pixels_to_image(pixel_string, size=48):

    pixel_values = np.array(pixel_string.split(), dtype="float32")
    image_2d = pixel_values.reshape(size, size)
    return image_2d

# ---- Pick ONE sample image per emotion class ----
emotion_order = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]

fig, axes = plt.subplots(1, 7, figsize=(20, 4))
fig.patch.set_facecolor('white')

# Distinct border color per emotion (matches our rainbow theme from Step 3)
colors = plt.cm.rainbow(np.linspace(0, 1, 7))

for idx, emotion in enumerate(emotion_order):
    # Get the first row matching this emotion
    sample_row = fer_df[fer_df["emotion_label"] == emotion].iloc[0]
    img = pixels_to_image(sample_row["pixels"])

    ax = axes[idx]
    ax.imshow(img, cmap="gray")
    ax.set_title(emotion, fontsize=13, fontweight="bold", color=colors[idx])
    ax.axis("off")

    # Colored border around each face to match our theme
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(colors[idx])
        spine.set_linewidth(4)

plt.suptitle("FER2013 — Sample Face for Each Emotion Class",
              fontsize=17, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

---
## STEP 5: PIXEL INTENSITY DISTRIBUTION — HISTOGRAM + KDE(kernel Density Estimation)
---

In [ ]:
import seaborn as sns

# ---- Sample a subset of images for speed (analyzing all 35k+ images pixel-by-pixel is heavy) ----
sample_size = 2000
sample_df = fer_df.sample(n=sample_size, random_state=SEED)

# ---- Convert all sampled pixel strings into a single flat array of pixel values ----
all_pixel_values = []
for pixel_string in sample_df["pixels"]:
    pixels = np.array(pixel_string.split(), dtype="float32")
    all_pixel_values.extend(pixels)

all_pixel_values = np.array(all_pixel_values)

# ---- Basic statistics (Statistics Module) ----
print("Pixel Intensity Statistics (sampled):")
print(f"  Mean     : {all_pixel_values.mean():.2f}")
print(f"  Std Dev  : {all_pixel_values.std():.2f}")
print(f"  Min      : {all_pixel_values.min():.2f}")
print(f"  Max      : {all_pixel_values.max():.2f}")
print(f"  Skewness : {pd.Series(all_pixel_values).skew():.3f}")
print(f"  Kurtosis : {pd.Series(all_pixel_values).kurt():.3f}")

# ---- Plot: Histogram + KDE overlay ----
plt.figure(figsize=(12, 6))

sns.histplot(
    all_pixel_values,
    bins=50,
    kde=True,
    color="#6C5CE7",
    edgecolor="white",
    alpha=0.75,
    line_kws={"linewidth": 3, "color": "#FD79A8"}
)

plt.title("FER2013 — Pixel Intensity Distribution (Histogram + KDE)",
          fontsize=16, fontweight="bold")
plt.xlabel("Pixel Intensity (0 = Black, 255 = White)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.axvline(all_pixel_values.mean(), color="#00B894", linestyle="--",
            linewidth=2, label=f"Mean = {all_pixel_values.mean():.1f}")
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## STEP 6: CLASS-WISE AVERAGE FACE VISUALIZATION
---

In [ ]:
def pixels_to_image(pixel_string, size=48):
    pixel_values = np.array(pixel_string.split(), dtype="float32")
    return pixel_values.reshape(size, size)

emotion_order = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]
colors = plt.cm.rainbow(np.linspace(0, 1, 7))

fig, axes = plt.subplots(1, 7, figsize=(20, 4))
fig.patch.set_facecolor('white')

average_faces = {}   

for idx, emotion in enumerate(emotion_order):
    # Get ALL rows for this emotion class
    class_rows = fer_df[fer_df["emotion_label"] == emotion]

    # Convert every image to 2D array, then stack into a 3D array (N, 48, 48)
    all_images = np.stack([pixels_to_image(p) for p in class_rows["pixels"]])

    # Compute the pixel-wise MEAN across all images in this class
    avg_face = all_images.mean(axis=0)
    average_faces[emotion] = avg_face

    ax = axes[idx]
    ax.imshow(avg_face, cmap="gray")
    ax.set_title(f"{emotion}\n(n={len(class_rows)})",
                 fontsize=12, fontweight="bold", color=colors[idx])
    ax.axis("off")

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(colors[idx])
        spine.set_linewidth(4)

plt.suptitle("FER2013 — Average Face per Emotion Class",
              fontsize=17, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

---
## STEP 7: CLASS IMBALANCE HANDLING — COMPUTE CLASS WEIGHTS
---

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_labels = np.sort(fer_df["emotion"].unique())

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=class_labels,
    y=fer_df["emotion"]
)

# ---- Build a dictionary
class_weight_dict = dict(zip(class_labels, class_weights_array))

# ---- Map back to readable emotion names for display ----
emotion_map = {0: "Angry", 1: "Disgust", 2: "Fear", 3: "Happy",
               4: "Sad", 5: "Surprise", 6: "Neutral"}

weights_df = pd.DataFrame({
    "Emotion": [emotion_map[i] for i in class_labels],
    "Count": [ (fer_df["emotion"] == i).sum() for i in class_labels ],
    "Weight": class_weights_array
}).sort_values("Weight", ascending=False)

print("Computed Class Weights:\n")
print(weights_df.to_string(index=False))

# ---- Visualization: Bar chart comparing Count vs Weight ----
fig, ax1 = plt.subplots(figsize=(12, 6))

colors = plt.cm.rainbow(np.linspace(0, 1, len(weights_df)))

# Bar chart for image counts
bars = ax1.bar(weights_df["Emotion"], weights_df["Count"], color=colors, alpha=0.8, edgecolor="white")
ax1.set_ylabel("Number of Images", fontsize=12, fontweight="bold")
ax1.set_xlabel("Emotion", fontsize=12)
ax1.set_title("Class Imbalance vs Computed Class Weights", fontsize=16, fontweight="bold")

# Overlay line for weights on secondary axis
ax2 = ax1.twinx()
ax2.plot(weights_df["Emotion"], weights_df["Weight"], color="black",
         marker="o", markersize=8, linewidth=2.5, label="Class Weight")
ax2.set_ylabel("Class Weight (Higher = More Important)", fontsize=12, fontweight="bold")

for i, (count, weight) in enumerate(zip(weights_df["Count"], weights_df["Weight"])):
    ax2.text(i, weight + 0.05, f"{weight:.2f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

print("\n✅ 'class_weight_dict' is now ready to be passed into model.fit(class_weight=...) later.")
print(class_weight_dict)

---
## STEP 8: PREPARE TRAIN / VALIDATION / TEST SPLITS
---

In [ ]:
from sklearn.model_selection import train_test_split

def pixels_to_image(pixel_string, size=48):
    """Convert a space-separated pixel string into a 2D numpy array."""
    pixel_values = np.array(pixel_string.split(), dtype="float32")
    return pixel_values.reshape(size, size)

print("Converting all pixel strings into image arrays... (this may take a minute)")

X_images = np.stack([pixels_to_image(p) for p in fer_df["pixels"]])
y_labels = fer_df["emotion"].values

print(f"Raw image tensor shape : {X_images.shape}")   # (N, 48, 48)
print(f"Labels shape           : {y_labels.shape}")   # (N,)


X_images_normalized = X_images / 255.0


X_images_final = X_images_normalized.reshape(-1, 48, 48, 1)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_images_final, y_labels,
    test_size=0.30,
    random_state=SEED,
    stratify=y_labels         
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

# ---- Summary ----
print("\n✅ Data Split Complete:")
print(f"  Training set   : {X_train.shape[0]} images  ({X_train.shape[0]/len(y_labels):.1%})")
print(f"  Validation set : {X_val.shape[0]} images  ({X_val.shape[0]/len(y_labels):.1%})")
print(f"  Test set       : {X_test.shape[0]} images  ({X_test.shape[0]/len(y_labels):.1%})")
print(f"\nFinal image tensor shape (per sample): {X_train.shape[1:]}")
print(f"Pixel value range after normalization  : [{X_train.min():.2f}, {X_train.max():.2f}]")

# ---- Visualization: Bar chart showing split sizes ----
import matplotlib.pyplot as plt

split_names = ["Train", "Validation", "Test"]
split_sizes = [X_train.shape[0], X_val.shape[0], X_test.shape[0]]
split_colors = ["#6C5CE7", "#00B894", "#FD79A8"]

plt.figure(figsize=(8, 5))
bars = plt.bar(split_names, split_sizes, color=split_colors, edgecolor="white", linewidth=2)

for bar, size in zip(bars, split_sizes):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
              f"{size:,}", ha="center", fontsize=11, fontweight="bold")

plt.title("FER2013 — Train / Validation / Test Split Sizes", fontsize=15, fontweight="bold")
plt.ylabel("Number of Images", fontsize=12)
plt.tight_layout()
plt.show()

---
## STEP 9: CLASSICAL MACHINE LEARNING MODELS — BASELINE COMPARISON
---

In [ ]:
import time
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat   = X_val.reshape(X_val.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)

print(f"Flattened shape: {X_train_flat.shape}  (was 48x48x1 = 2304 features per image)")

subset_size = 6000
rng = np.random.RandomState(SEED)
subset_idx = rng.choice(len(X_train_flat), size=subset_size, replace=False)

X_train_sub = X_train_flat[subset_idx]
y_train_sub = y_train[subset_idx]

pca = PCA(n_components=100, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_sub)
X_val_pca   = pca.transform(X_val_flat)
X_test_pca  = pca.transform(X_test_flat)

print(f"Variance retained by 100 PCA components: {pca.explained_variance_ratio_.sum():.2%}")

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Decision Tree":        DecisionTreeClassifier(max_depth=15, random_state=SEED),
    "Random Forest":        RandomForestClassifier(n_estimators=150, max_depth=15, random_state=SEED, n_jobs=-1),
    "SVM (RBF Kernel)":     SVC(kernel="rbf", random_state=SEED),
    "Naive Bayes":          GaussianNB()
}

results = []

for name, model in models.items():
    start_time = time.time()
    model.fit(X_train_pca, y_train_sub)
    train_time = time.time() - start_time

    val_preds = model.predict(X_val_pca)
    val_acc = accuracy_score(y_val, val_preds)

    results.append({"Model": name, "Validation Accuracy": val_acc, "Training Time (s)": train_time})
    print(f"{name:22s} | Val Accuracy: {val_acc:.4f} | Training Time: {train_time:.2f}s")

results_df = pd.DataFrame(results).sort_values("Validation Accuracy", ascending=False)

# ---- Visualization — Model Comparison Bar Chart ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.plasma(np.linspace(0.15, 0.9, len(results_df)))

# Accuracy comparison
axes[0].barh(results_df["Model"], results_df["Validation Accuracy"], color=colors, edgecolor="white", linewidth=2)
axes[0].set_xlabel("Validation Accuracy", fontsize=12, fontweight="bold")
axes[0].set_title("Classical ML Models — Accuracy Comparison", fontsize=14, fontweight="bold")
for i, v in enumerate(results_df["Validation Accuracy"]):
    axes[0].text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=10, fontweight="bold")

# Training time comparison
axes[1].barh(results_df["Model"], results_df["Training Time (s)"], color=colors, edgecolor="white", linewidth=2)
axes[1].set_xlabel("Training Time (seconds)", fontsize=12, fontweight="bold")
axes[1].set_title("Classical ML Models — Training Time Comparison", fontsize=14, fontweight="bold")
for i, v in enumerate(results_df["Training Time (s)"]):
    axes[1].text(v + 0.5, i, f"{v:.1f}s", va="center", fontsize=10, fontweight="bold")

plt.suptitle("FER2013 — Classical Machine Learning Baseline Results", fontsize=17, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

---
## STEP 10: CONFUSION MATRIX — 3D BAR VISUALIZATION
---

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from sklearn.metrics import confusion_matrix

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"Best classical model: {best_model_name}")

test_preds = best_model.predict(X_test_pca)

emotion_order = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]
cm = confusion_matrix(y_test, test_preds)

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")

n_classes = len(emotion_order)
xpos, ypos = np.meshgrid(np.arange(n_classes), np.arange(n_classes))
xpos = xpos.flatten()
ypos = ypos.flatten()
zpos = np.zeros_like(xpos)

dx = dy = 0.7
dz = cm.flatten()

colors = plt.cm.plasma(dz / dz.max())

ax.bar3d(xpos, ypos, zpos, dx, dy, dz, color=colors, edgecolor="white", linewidth=0.5, shade=True)

ax.set_xticks(np.arange(n_classes) + dx/2)
ax.set_xticklabels(emotion_order, fontsize=10, fontweight="bold", rotation=45, ha="right")
ax.set_yticks(np.arange(n_classes) + dy/2)
ax.set_yticklabels(emotion_order, fontsize=10, fontweight="bold")

ax.set_xlabel("Predicted Emotion", fontsize=12, fontweight="bold", labelpad=15)
ax.set_ylabel("True Emotion", fontsize=12, fontweight="bold", labelpad=15)
ax.set_zlabel("Count", fontsize=12, fontweight="bold")

ax.set_title(f"3D Confusion Matrix — {best_model_name}",
             fontsize=17, fontweight="bold", pad=20)

ax.view_init(elev=25, azim=-45)  

plt.tight_layout()
plt.show()

# ---- Print key insight - most confused pair ----
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
max_confusion_idx = np.unravel_index(np.argmax(cm_no_diag), cm_no_diag.shape)
print(f"\nMost confused pair: True '{emotion_order[max_confusion_idx[0]]}' "
      f"predicted as '{emotion_order[max_confusion_idx[1]]}' "
      f"({cm_no_diag[max_confusion_idx]} times)")

In [ ]:
cm = confusion_matrix(y_test, test_preds)

# ---- Plot as heatmap ----
plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,          
    fmt="d",             
    cmap="Blues",
    xticklabels=emotion_order,
    yticklabels=emotion_order,
    cbar=True
)

plt.title(f"Confusion Matrix — {best_model_name}", fontsize=15, fontweight="bold")
plt.xlabel("Predicted Emotion", fontsize=12)
plt.ylabel("True Emotion", fontsize=12)
plt.tight_layout()
plt.show()

---
## STEP 11: SIMPLE ANN (ARTIFICIAL NEURAL NETWORK)
---

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt


X_train_ann = X_train.reshape(X_train.shape[0], -1)   
X_val_ann   = X_val.reshape(X_val.shape[0], -1)
X_test_ann  = X_test.reshape(X_test.shape[0], -1)

print(f"ANN input shape: {X_train_ann.shape}")


ann_model = models.Sequential([
    layers.Input(shape=(2304,)),
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dense(7, activation="softmax")   
])

ann_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

ann_model.summary()


history = ann_model.fit(
    X_train_ann, y_train,
    validation_data=(X_val_ann, y_val),
    epochs=25,
    batch_size=128,
    class_weight=class_weight_dict,
    verbose=1
)

test_loss, test_acc = ann_model.evaluate(X_test_ann, y_test, verbose=0)
print(f"\n✅ ANN Test Accuracy: {test_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history.history["accuracy"], color="#6C5CE7", linewidth=2.5, marker="o", markersize=3, label="Train Accuracy")
axes[0].plot(history.history["val_accuracy"], color="#FD79A8", linewidth=2.5, marker="o", markersize=3, label="Validation Accuracy")
axes[0].set_title("ANN — Accuracy Curve", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history["loss"], color="#00B894", linewidth=2.5, marker="o", markersize=3, label="Train Loss")
axes[1].plot(history.history["val_loss"], color="#E17055", linewidth=2.5, marker="o", markersize=3, label="Validation Loss")
axes[1].set_title("ANN — Loss Curve", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("ANN Training Performance — FER2013", fontsize=17, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

---
## STEP 12: CNN (CONVOLUTIONAL NEURAL NETWORK)
---

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

print(f"CNN input shape: {X_train.shape[1:]}")   

cnn_model = models.Sequential([
    layers.Input(shape=(48, 48, 1)),

    # Convolution Block 1
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Convolution Block 2
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Convolution Block 3
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Fully Connected Head
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(7, activation="softmax")   # 7 emotion classes
])

cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

cnn_history = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    class_weight=class_weight_dict,
    verbose=1
)

cnn_test_loss, cnn_test_acc = cnn_model.evaluate(X_test, y_test, verbose=0)
print(f"\n✅ CNN Test Accuracy: {cnn_test_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(cnn_history.history["accuracy"], color="#0984E3", linewidth=2.5, marker="o", markersize=3, label="Train Accuracy")
axes[0].plot(cnn_history.history["val_accuracy"], color="#FDCB6E", linewidth=2.5, marker="o", markersize=3, label="Validation Accuracy")
axes[0].set_title("CNN — Accuracy Curve", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(cnn_history.history["loss"], color="#00B894", linewidth=2.5, marker="o", markersize=3, label="Train Loss")
axes[1].plot(cnn_history.history["val_loss"], color="#D63031", linewidth=2.5, marker="o", markersize=3, label="Validation Loss")
axes[1].set_title("CNN — Loss Curve", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("CNN Training Performance — FER2013", fontsize=17, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

---
## STEP 13: CNN CONFUSION MATRIX
---

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cnn_test_probs = cnn_model.predict(X_test)
cnn_test_preds = np.argmax(cnn_test_probs, axis=1)   

emotion_order = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral"]
cnn_cm = confusion_matrix(y_test, cnn_test_preds)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cnn_cm,
    annot=True,
    fmt="d",
    cmap="Purples",
    xticklabels=emotion_order,
    yticklabels=emotion_order,
    cbar=True
)
plt.title("CNN — Confusion Matrix (Test Set)", fontsize=15, fontweight="bold")
plt.xlabel("Predicted Emotion", fontsize=12)
plt.ylabel("True Emotion", fontsize=12)
plt.tight_layout()
plt.show()

print("CNN Classification Report:\n")
print(classification_report(y_test, cnn_test_preds, target_names=emotion_order))

sad_idx = emotion_order.index("Sad")
happy_idx = emotion_order.index("Happy")
disgust_idx = emotion_order.index("Disgust")

print(f"\n🔍 Direct Comparison:")
print(f"  'Sad' correctly predicted as 'Sad'   : {cnn_cm[sad_idx, sad_idx]}")
print(f"  'Sad' wrongly predicted as 'Happy'   : {cnn_cm[sad_idx, happy_idx]}")
print(f"  'Disgust' correctly predicted at all : {cnn_cm[disgust_idx, disgust_idx]} (out of {cnn_cm[disgust_idx].sum()} total)")

---
## STEP 14: LSTM MODEL ON FER2013 (SEQUENCE-BASED APPROACH)
---

In [ ]:
from tensorflow.keras import layers, models
X_train_seq = X_train.reshape(X_train.shape[0], 48, 48)   # (N, timesteps=48, features=48)
X_val_seq   = X_val.reshape(X_val.shape[0], 48, 48)
X_test_seq  = X_test.reshape(X_test.shape[0], 48, 48)

print(f"LSTM input shape: {X_train_seq.shape}  (samples, timesteps, features)")

lstm_model = models.Sequential([
    layers.Input(shape=(48, 48)),

    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.3),

    layers.LSTM(64),
    layers.Dropout(0.3),

    layers.Dense(64, activation="relu"),
    layers.Dense(7, activation="softmax")   # 7 emotion classes
])

lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

lstm_history = lstm_model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=25,
    batch_size=128,
    class_weight=class_weight_dict,
    verbose=1
)

lstm_test_loss, lstm_test_acc = lstm_model.evaluate(X_test_seq, y_test, verbose=0)
print(f"\n✅ LSTM Test Accuracy: {lstm_test_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(lstm_history.history["accuracy"], color="#E84393", linewidth=2.5, marker="o", markersize=3, label="Train Accuracy")
axes[0].plot(lstm_history.history["val_accuracy"], color="#00CEC9", linewidth=2.5, marker="o", markersize=3, label="Validation Accuracy")
axes[0].set_title("LSTM — Accuracy Curve", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(lstm_history.history["loss"], color="#FDCB6E", linewidth=2.5, marker="o", markersize=3, label="Train Loss")
axes[1].plot(lstm_history.history["val_loss"], color="#6C5CE7", linewidth=2.5, marker="o", markersize=3, label="Validation Loss")
axes[1].set_title("LSTM — Loss Curve", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("LSTM Training Performance — FER2013", fontsize=17, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

---
## STEP 16: FINAL MODEL COMPARISON — FER2013 SUMMARY
---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

best_classical_acc = accuracy_score(y_test, best_model.predict(X_test_pca))

comparison_df = pd.DataFrame({
    "Model": [f"Best Classical ML\n({best_model_name})", "ANN", "CNN", "LSTM"],
    "Test Accuracy": [best_classical_acc, test_acc, cnn_test_acc, lstm_test_acc],
    "Category": ["Classical ML", "Deep Learning", "Deep Learning", "Deep Learning"]
})

print("Final Model Comparison — FER2013:\n")
print(comparison_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 7))

bar_colors = ["#B2BEC3", "#6C5CE7", "#00B894", "#E84393"]

bars = ax.bar(
    comparison_df["Model"],
    comparison_df["Test Accuracy"],
    color=bar_colors,
    edgecolor="white",
    linewidth=2.5,
    width=0.6
)

for bar, acc in zip(bars, comparison_df["Test Accuracy"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{acc:.1%}", ha="center", fontsize=13, fontweight="bold")

best_idx = comparison_df["Test Accuracy"].idxmax()
ax.text(best_idx, comparison_df["Test Accuracy"].iloc[best_idx] + 0.06, "⭐ Best",
        ha="center", fontsize=13, fontweight="bold", color="#D63031")

ax.set_ylabel("Test Accuracy", fontsize=13, fontweight="bold")
ax.set_title("FER2013 — Final Model Comparison\n(Classical ML vs Deep Learning)",
             fontsize=16, fontweight="bold", pad=15)
ax.set_ylim(0, max(comparison_df["Test Accuracy"]) + 0.15)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

improvement = (comparison_df["Test Accuracy"].max() - best_classical_acc) / best_classical_acc
print(f"\n📊 Key Insight: Best deep learning model improves accuracy by "
      f"{improvement:.1%} over the best classical ML model ({best_model_name}).")

---
## STEP 17: TRANSFER LEARNING — MOBILENETV2
---

In [ ]:
from tensorflow.keras.applications import MobileNetV2
import matplotlib.pyplot as plt

IMG_SIZE = 96

def prepare_for_mobilenet(images):
    """Convert grayscale batch to resized RGB batch for MobileNetV2."""
    images_rgb = tf.image.grayscale_to_rgb(tf.constant(images))          # (N,48,48,1) -> (N,48,48,3)
    images_resized = tf.image.resize(images_rgb, [IMG_SIZE, IMG_SIZE])   # -> (N,96,96,3)
    return images_resized.numpy()

print("Converting and resizing images for MobileNetV2... (this takes a moment)")
X_train_mnv2 = prepare_for_mobilenet(X_train)
X_val_mnv2   = prepare_for_mobilenet(X_val)
X_test_mnv2  = prepare_for_mobilenet(X_test)

print(f"New shape for MobileNetV2 input: {X_train_mnv2.shape}")

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,      
    weights="imagenet"       
)

base_model.trainable = False

mobilenet_model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(7, activation="softmax")
])

mobilenet_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

mobilenet_model.summary()

mobilenet_history = mobilenet_model.fit(
    X_train_mnv2, y_train,
    validation_data=(X_val_mnv2, y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weight_dict,
    verbose=1
)

mnv2_test_loss, mnv2_test_acc = mobilenet_model.evaluate(X_test_mnv2, y_test, verbose=0)
print(f"\n✅ MobileNetV2 Test Accuracy: {mnv2_test_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(mobilenet_history.history["accuracy"], color="#00B894", linewidth=2.5, marker="o", markersize=3, label="Train Accuracy")
axes[0].plot(mobilenet_history.history["val_accuracy"], color="#FDCB6E", linewidth=2.5, marker="o", markersize=3, label="Validation Accuracy")
axes[0].set_title("MobileNetV2 — Accuracy Curve", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(mobilenet_history.history["loss"], color="#0984E3", linewidth=2.5, marker="o", markersize=3, label="Train Loss")
axes[1].plot(mobilenet_history.history["val_loss"], color="#D63031", linewidth=2.5, marker="o", markersize=3, label="Validation Loss")
axes[1].set_title("MobileNetV2 — Loss Curve", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("MobileNetV2 Transfer Learning — FER2013", fontsize=17, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

---
## STEP 18: TRANSFER LEARNING — EFFICIENTNETB0
---

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
import matplotlib.pyplot as plt

print(f"Reusing prepared RGB images, shape: {X_train_mnv2.shape}")

efficientnet_base = EfficientNetB0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

efficientnet_base.trainable = False

efficientnet_model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    efficientnet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(7, activation="softmax")
])

efficientnet_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

efficientnet_model.summary()

efficientnet_history = efficientnet_model.fit(
    X_train_mnv2, y_train,
    validation_data=(X_val_mnv2, y_val),
    epochs=15,
    batch_size=64,
    class_weight=class_weight_dict,
    verbose=1
)

effnet_test_loss, effnet_test_acc = efficientnet_model.evaluate(X_test_mnv2, y_test, verbose=0)
print(f"\n✅ EfficientNetB0 Test Accuracy: {effnet_test_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(efficientnet_history.history["accuracy"], color="#00CEC9", linewidth=2.5, marker="o", markersize=3, label="Train Accuracy")
axes[0].plot(efficientnet_history.history["val_accuracy"], color="#E17055", linewidth=2.5, marker="o", markersize=3, label="Validation Accuracy")
axes[0].set_title("EfficientNetB0 — Accuracy Curve", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(efficientnet_history.history["loss"], color="#6C5CE7", linewidth=2.5, marker="o", markersize=3, label="Train Loss")
axes[1].plot(efficientnet_history.history["val_loss"], color="#FDCB6E", linewidth=2.5, marker="o", markersize=3, label="Validation Loss")
axes[1].set_title("EfficientNetB0 — Loss Curve", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("EfficientNetB0 Transfer Learning — FER2013", fontsize=17, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

print(f"\n🔍 MobileNetV2 Test Accuracy   : {mnv2_test_acc:.4f}")
print(f"🔍 EfficientNetB0 Test Accuracy: {effnet_test_acc:.4f}")

............................................................................................................................................................................
---
# STEP 19: COMBINED FINAL SUMMARY — ALL FER2013 MODELS
............................................................................................................................................................................
---

In [ ]:
final_summary_df = pd.DataFrame({
    "Model": [
        f"Classical ML\n({best_model_name})",
        "ANN",
        "CNN",
        "LSTM",
        "MobileNetV2\n(Transfer Learning)",
        "EfficientNetB0\n(Transfer Learning)"
    ],
    "Test Accuracy": [
        best_classical_acc,
        test_acc,
        cnn_test_acc,
        lstm_test_acc,
        mnv2_test_acc,
        effnet_test_acc
    ],
    "Category": [
        "Classical ML", "Deep Learning", "Deep Learning",
        "Deep Learning", "Transfer Learning", "Transfer Learning"
    ]
}).sort_values("Test Accuracy", ascending=True).reset_index(drop=True)

print("FINAL SUMMARY — ALL MODELS (FER2013):\n")
print(final_summary_df.to_string(index=False))

category_colors = {
    "Classical ML": "#B2BEC3",
    "Deep Learning": "#6C5CE7",
    "Transfer Learning": "#00B894"
}
bar_colors = [category_colors[cat] for cat in final_summary_df["Category"]]

fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(
    final_summary_df["Model"],
    final_summary_df["Test Accuracy"],
    color=bar_colors,
    edgecolor="white",
    linewidth=2.5
)

for bar, acc in zip(bars, final_summary_df["Test Accuracy"]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{acc:.1%}", va="center", fontsize=12, fontweight="bold")

best_row = final_summary_df.iloc[-1]
ax.text(best_row["Test Accuracy"] + 0.09, len(final_summary_df) - 1,
        "⭐", va="center", fontsize=18)

ax.set_xlabel("Test Accuracy", fontsize=13, fontweight="bold")
ax.set_title("FER2013 — Complete Model Comparison\n(Classical ML → Deep Learning → Transfer Learning)",
              fontsize=16, fontweight="bold", pad=15)
ax.set_xlim(0, max(final_summary_df["Test Accuracy"]) + 0.15)
ax.grid(axis="x", alpha=0.3)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=cat) for cat, color in category_colors.items()]
ax.legend(handles=legend_elements, loc="lower right", fontsize=11)

plt.tight_layout()
plt.show()

best_model_overall = final_summary_df.iloc[-1]["Model"].replace("\n", " ")
worst_model_overall = final_summary_df.iloc[0]["Model"].replace("\n", " ")
print(f"\n📊 FINAL CONCLUSION (Facial Emotion Recognition):")
print(f"   Best performing model  : {best_model_overall} ({final_summary_df.iloc[-1]['Test Accuracy']:.1%})")
print(f"   Weakest performing model: {worst_model_overall} ({final_summary_df.iloc[0]['Test Accuracy']:.1%})")

---
## STEP 20: LOAD AND EXPLORE RAVDESS DATASET
---

In [ ]:
RAVDESS_PATH = "/kaggle/input/datasets/uwrfkaggler/ravdess-emotional-speech-audio/audio_speech_actors_01-24"

# ---- RAVDESS filename format: 03-01-05-01-02-01-12.wav ----
# Position 3 (index 2) = EMOTION CODE:
# 01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised

ravdess_emotion_map = {
    "01": "Neutral", "02": "Calm", "03": "Happy", "04": "Sad",
    "05": "Angry", "06": "Fear", "07": "Disgust", "08": "Surprise"
}

file_paths = []
emotions = []
actors = []

for actor_folder in os.listdir(RAVDESS_PATH):
    actor_path = os.path.join(RAVDESS_PATH, actor_folder)
    if not os.path.isdir(actor_path):
        continue

    for filename in os.listdir(actor_path):
        if filename.endswith(".wav"):
            parts = filename.split("-")
            emotion_code = parts[2]                      # 3rd part = emotion code
            emotion_label = ravdess_emotion_map[emotion_code]

            file_paths.append(os.path.join(actor_path, filename))
            emotions.append(emotion_label)
            actors.append(actor_folder)

ravdess_df = pd.DataFrame({
    "file_path": file_paths,
    "emotion": emotions,
    "actor": actors
})

print(f"Total audio files found: {len(ravdess_df)}")
print(f"\nFirst 5 rows:")
display(ravdess_df.head())

print(f"\nEmotion distribution:")
print(ravdess_df["emotion"].value_counts())

import matplotlib.pyplot as plt
import numpy as np

emotion_counts = ravdess_df["emotion"].value_counts()
labels = emotion_counts.index.tolist()
values = emotion_counts.values
n = len(labels)
colors = plt.cm.rainbow(np.linspace(0, 1, n))

fig = plt.figure(figsize=(9, 9))
ax = fig.add_subplot(111, polar=True)

angles = np.linspace(0, 2*np.pi, n, endpoint=False)
bars = ax.bar(angles, values, width=(2*np.pi/n)*0.85, color=colors, edgecolor="white", linewidth=2, alpha=0.9)

ax.set_theta_offset(np.pi/2)
ax.set_theta_direction(-1)
ax.set_xticks(angles)
ax.set_xticklabels(labels, fontsize=11, fontweight="bold")
ax.set_yticklabels([])
ax.spines["polar"].set_visible(False)

for angle, value in zip(angles, values):
    ax.text(angle, value + max(values)*0.08, f"{value}", ha="center", fontsize=10, fontweight="bold")

plt.title("RAVDESS — Emotion Distribution (Audio Files)", fontsize=16, fontweight="bold", pad=30)
plt.tight_layout()
plt.show()

---
## STEP 21: AUDIO WAVEFORM VISUALIZATION
---

In [ ]:
import librosa
import librosa.display

emotion_order_audio = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral", "Calm"]
colors = plt.cm.rainbow(np.linspace(0, 1, len(emotion_order_audio)))

fig, axes = plt.subplots(4, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, emotion in enumerate(emotion_order_audio):
  
    sample_file = ravdess_df[ravdess_df["emotion"] == emotion].iloc[0]["file_path"]


    audio_signal, sample_rate = librosa.load(sample_file, sr=None)


    ax = axes[idx]
    librosa.display.waveshow(audio_signal, sr=sample_rate, ax=ax, color=colors[idx])
    ax.set_title(f"{emotion} — Waveform", fontsize=12, fontweight="bold", color=colors[idx])
    ax.set_xlabel("Time (seconds)", fontsize=9)
    ax.set_ylabel("Amplitude", fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("RAVDESS — Audio Waveforms Across Emotions", fontsize=18, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"Sample rate: {sample_rate} Hz")
print(f"Example audio duration: {len(audio_signal)/sample_rate:.2f} seconds")

---
## STEP 22: MFCC (Mel-Frequency Cepstral Coefficients) EXTRACTION AND VISUALIZATION
---

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

emotion_order_audio = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral", "Calm"]

fig, axes = plt.subplots(4, 2, figsize=(16, 16))
axes = axes.flatten()

n_mfcc = 40  

for idx, emotion in enumerate(emotion_order_audio):
    sample_file = ravdess_df[ravdess_df["emotion"] == emotion].iloc[0]["file_path"]

    # ---- Load audio ----
    audio_signal, sample_rate = librosa.load(sample_file, sr=None)

    # ---- Extract MFCC features ----
    mfcc = librosa.feature.mfcc(y=audio_signal, sr=sample_rate, n_mfcc=n_mfcc)

    # ---- Plot as a heatmap ----
    ax = axes[idx]
    img = librosa.display.specshow(mfcc, sr=sample_rate, x_axis="time", ax=ax, cmap="magma")
    ax.set_title(f"{emotion} — MFCC Heatmap", fontsize=12, fontweight="bold")
    ax.set_ylabel("MFCC Coefficients", fontsize=9)
    fig.colorbar(img, ax=ax, format="%+2.0f")

plt.suptitle("RAVDESS — MFCC Heatmaps Across Emotions", fontsize=18, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"MFCC shape for one sample: {mfcc.shape}  (n_mfcc={n_mfcc}, time_frames={mfcc.shape[1]})")

---
## STEP 23: SPECTROGRAM VISUALIZATION
---

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

emotion_order_audio = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral", "Calm"]

fig, axes = plt.subplots(4, 2, figsize=(16, 16))
axes = axes.flatten()

for idx, emotion in enumerate(emotion_order_audio):
    sample_file = ravdess_df[ravdess_df["emotion"] == emotion].iloc[0]["file_path"]

    # ---- Load audio ----
    audio_signal, sample_rate = librosa.load(sample_file, sr=None)

    stft_result = librosa.stft(audio_signal)

    spectrogram_db = librosa.amplitude_to_db(np.abs(stft_result), ref=np.max)

    ax = axes[idx]
    img = librosa.display.specshow(
        spectrogram_db, sr=sample_rate,
        x_axis="time", y_axis="hz", ax=ax, cmap="inferno"
    )
    ax.set_title(f"{emotion} — Spectrogram", fontsize=12, fontweight="bold")
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

plt.suptitle("RAVDESS — Spectrograms Across Emotions", fontsize=18, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print(f"Spectrogram shape for one sample: {spectrogram_db.shape}  (frequency_bins, time_frames)")

---
## STEP 24: MFCC EXTRACTION + ML/LSTM COMPARISON
---

In [ ]:
import librosa, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from tensorflow.keras import layers, models

print("Extracting MFCC features for all files... (2-3 minutes)")
features, labels = [], []

for _, row in ravdess_df.iterrows():
    y_audio, sr = librosa.load(row["file_path"], sr=22050)  # lower sr = faster
    mfcc = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=40)
    features.append(np.mean(mfcc.T, axis=0))   
    labels.append(row["emotion"])

X_audio = np.array(features)
le = LabelEncoder()
y_audio = le.fit_transform(labels)
print(f"Feature matrix shape: {X_audio.shape}")

X_tr, X_te, y_tr, y_te = train_test_split(X_audio, y_audio, test_size=0.2, random_state=42, stratify=y_audio)
scaler = StandardScaler()
X_tr_s, X_te_s = scaler.fit_transform(X_tr), scaler.transform(X_te)

quick_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42),
    "SVM": SVC(kernel="rbf")
}
audio_results = {}
for name, m in quick_models.items():
    m.fit(X_tr_s, y_tr)
    audio_results[name] = accuracy_score(y_te, m.predict(X_te_s))

max_len = 130 
def get_mfcc_seq(path):
    y_audio, sr = librosa.load(path, sr=22050)
    mfcc = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=40).T  # (time, 40)
    if mfcc.shape[0] < max_len:
        mfcc = np.pad(mfcc, ((0, max_len - mfcc.shape[0]), (0,0)))
    else:
        mfcc = mfcc[:max_len]
    return mfcc

print("Preparing sequences for LSTM...")
X_seq = np.array([get_mfcc_seq(p) for p in ravdess_df["file_path"]])
X_seq_tr, X_seq_te, y_seq_tr, y_seq_te = train_test_split(X_seq, y_audio, test_size=0.2, random_state=42, stratify=y_audio)

lstm_audio = models.Sequential([
    layers.Input(shape=(max_len, 40)),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(64),
    layers.Dense(64, activation="relu"),
    layers.Dense(len(le.classes_), activation="softmax")
])
lstm_audio.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
lstm_audio.fit(X_seq_tr, y_seq_tr, validation_split=0.1, epochs=20, batch_size=32, verbose=0)
audio_results["LSTM"] = lstm_audio.evaluate(X_seq_te, y_seq_te, verbose=0)[1]

res_df = pd.DataFrame(list(audio_results.items()), columns=["Model", "Accuracy"]).sort_values("Accuracy")
colors = ["#B2BEC3", "#B2BEC3", "#B2BEC3", "#E84393"]

plt.figure(figsize=(10,6))
bars = plt.barh(res_df["Model"], res_df["Accuracy"], color=colors, edgecolor="white", linewidth=2)
for bar, acc in zip(bars, res_df["Accuracy"]):
    plt.text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2, f"{acc:.1%}", va="center", fontweight="bold")
plt.title("RAVDESS — Model Comparison (Classical ML vs LSTM)", fontsize=15, fontweight="bold")
plt.xlabel("Test Accuracy")
plt.tight_layout()
plt.show()

print("\n✅ Best model:", res_df.iloc[-1]["Model"], f"({res_df.iloc[-1]['Accuracy']:.1%})")

............................................................................................................................
---
## STEP 25: GOEMOTIONS — LOAD, PREPROCESS, ML/LSTM COMPARISON
............................................................................................................................
---

In [ ]:
plt.style.use("Solarize_Light2")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["legend.frameon"] = True
plt.rcParams["grid.alpha"] = 0.4

print("✅ Global plot style updated to 'Solarize_Light2'. All future graphs will use this theme automatically.")

---
## STEP 26: MAP EMOTION LABELS + COMBINE GOEMOTIONS SPLITS
---

In [ ]:
GOEMOTIONS_DIR = "/kaggle/input/datasets/debarshichanda/goemotions/data"

with open(os.path.join(GOEMOTIONS_DIR, "emotions.txt"), "r") as f:
    emotion_names = [line.strip() for line in f.readlines()]

print(f"Total emotion categories: {len(emotion_names)}")
print(emotion_names)

col_names = ["text", "emotion_ids", "comment_id"]

train_df = pd.read_csv(os.path.join(GOEMOTIONS_DIR, "train.tsv"), sep="\t", header=None, names=col_names)
dev_df   = pd.read_csv(os.path.join(GOEMOTIONS_DIR, "dev.tsv"),   sep="\t", header=None, names=col_names)
test_df  = pd.read_csv(os.path.join(GOEMOTIONS_DIR, "test.tsv"),  sep="\t", header=None, names=col_names)

print(f"\nTrain: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")

go_df = pd.concat([train_df, dev_df, test_df], ignore_index=True)
print(f"Combined total: {len(go_df)}")

go_df["primary_emotion_id"] = go_df["emotion_ids"].apply(lambda x: int(str(x).split(",")[0]))
go_df["emotion_label"] = go_df["primary_emotion_id"].apply(lambda idx: emotion_names[idx])

print("\nSample rows with mapped labels:")
display(go_df[["text", "emotion_label"]].head(10))

print("\nEmotion distribution (top 10):")
print(go_df["emotion_label"].value_counts().head(10))

---
## STEP 27: TEXT CLEANING + WORD CLOUD VISUALIZATION
---

In [ ]:
import re

# ---- List of common profanity/offensive words to remove (extend if needed) ----
profanity_list = [
    "fuck", "fucking", "fucked", "shit", "shitty", "ass", "asshole",
    "bitch", "damn", "bastard", "crap", "dick", "piss"
]

def clean_text_safe(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\[NAME\]|\[RELIGION\]", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    # ---- Remove profanity words ----
    words = text.split()
    words = [w for w in words if w not in profanity_list]
    text = " ".join(words)
    
    return text

# ---- Re-apply cleaning with the profanity filter ----
go_df["clean_text"] = go_df["text"].apply(clean_text_safe)

print("✅ Profanity filtered. Regenerating word clouds...\n")

import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS

emotions_to_plot = ["joy", "anger", "sadness", "fear"]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for idx, emotion in enumerate(emotions_to_plot):
    text_combined = " ".join(go_df[go_df["emotion_label"] == emotion]["clean_text"])
    wordcloud = WordCloud(
        width=800, height=500, background_color="white",
        colormap="plasma", stopwords=STOPWORDS, max_words=100
    ).generate(text_combined)
    axes[idx].imshow(wordcloud, interpolation="bilinear")
    axes[idx].set_title(f"Word Cloud — '{emotion.capitalize()}'", fontsize=14, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("GoEmotions — Word Clouds Across Key Emotions (Filtered)", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## STEP 28: TOKEN LENGTH DISTRIBUTION
---

In [ ]:
go_df["token_length"] = go_df["clean_text"].apply(lambda x: len(x.split()))

print("Token Length Statistics:")
print(go_df["token_length"].describe())

plt.figure(figsize=(11, 6))
sns.histplot(go_df["token_length"], bins=40, kde=True, color="#6C5CE7", edgecolor="white")

percentile_95 = int(go_df["token_length"].quantile(0.95))
plt.axvline(percentile_95, color="#D63031", linestyle="--", linewidth=2,
            label=f"95th Percentile = {percentile_95} words")

plt.title("GoEmotions — Comment Length Distribution (in Words)", fontsize=15, fontweight="bold")
plt.xlabel("Number of Words per Comment", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(fontsize=11)
plt.xlim(0, 60)
plt.tight_layout()
plt.show()

print(f"\n✅ Recommended max sequence length for tokenizer: {percentile_95} (covers 95% of comments)")

---
## STEP 29: TOKENIZATION + WORD EMBEDDINGS SETUP
---

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder

VOCAB_SIZE = 10000     
MAX_LEN = 30            
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(go_df["clean_text"])

print(f"Total unique words found : {len(tokenizer.word_index)}")
print(f"Vocabulary size used     : {VOCAB_SIZE}")

sequences = tokenizer.texts_to_sequences(go_df["clean_text"])

X_text_padded = pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")

print(f"\nPadded sequence matrix shape: {X_text_padded.shape}")

sample_idx = 0
print(f"\nOriginal text : {go_df['clean_text'].iloc[sample_idx]}")
print(f"Token sequence: {sequences[sample_idx]}")
print(f"Padded (len={MAX_LEN}): {X_text_padded[sample_idx]}")

le_text = LabelEncoder()
y_text = le_text.fit_transform(go_df["emotion_label"])

print(f"\nTotal emotion classes: {len(le_text.classes_)}")
print(f"Classes: {list(le_text.classes_)}")

---
## STEP 30: CLASSICAL ML MODELS ON TEXT
---

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(go_df["clean_text"])

print(f"TF-IDF feature matrix shape: {X_tfidf.shape}")

X_train_txt, X_test_txt, y_train_txt, y_test_txt = train_test_split(
    X_tfidf, y_text, test_size=0.2, random_state=SEED, stratify=y_text
)

text_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(max_iter=2000)
}

text_results = {}
for name, model in text_models.items():
    model.fit(X_train_txt, y_train_txt)
    preds = model.predict(X_test_txt)
    acc = accuracy_score(y_test_txt, preds)
    text_results[name] = acc
    print(f"{name:22s} | Test Accuracy: {acc:.4f}")

results_df_text = pd.DataFrame(list(text_results.items()), columns=["Model", "Accuracy"]).sort_values("Accuracy")

plt.figure(figsize=(10, 6))
colors = plt.cm.plasma(np.linspace(0.2, 0.8, len(results_df_text)))
bars = plt.barh(results_df_text["Model"], results_df_text["Accuracy"], color=colors, edgecolor="white", linewidth=2)

for bar, acc in zip(bars, results_df_text["Accuracy"]):
    plt.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f"{acc:.1%}", va="center", fontweight="bold")

plt.title("GoEmotions — Classical ML Models (TF-IDF Features)", fontsize=15, fontweight="bold")
plt.xlabel("Test Accuracy")
plt.tight_layout()
plt.show()

---
## STEP 31: LSTM MODEL WITH EMBEDDING LAYER — TEXT EMOTION CLASSIFICATION
---

In [ ]:
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

X_train_lstm, X_temp_lstm, y_train_lstm, y_temp_lstm = train_test_split(
    X_text_padded, y_text, test_size=0.30, random_state=SEED, stratify=y_text
)
X_val_lstm, X_test_lstm, y_val_lstm, y_test_lstm = train_test_split(
    X_temp_lstm, y_temp_lstm, test_size=0.50, random_state=SEED, stratify=y_temp_lstm
)

print(f"Train: {X_train_lstm.shape} | Val: {X_val_lstm.shape} | Test: {X_test_lstm.shape}")
EMBEDDING_DIM = 100
NUM_CLASSES = len(le_text.classes_)   # 28

text_lstm_model = models.Sequential([
    layers.Input(shape=(MAX_LEN,)),

    layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),

    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(64),
    layers.Dropout(0.3),

    layers.Dense(64, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

text_lstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

text_lstm_model.summary()

text_lstm_history = text_lstm_model.fit(
    X_train_lstm, y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    epochs=15,
    batch_size=128,
    verbose=1
)

lstm_text_loss, lstm_text_acc = text_lstm_model.evaluate(X_test_lstm, y_test_lstm, verbose=0)
print(f"\n✅ Text LSTM Test Accuracy: {lstm_text_acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(text_lstm_history.history["accuracy"], color="#6C5CE7", linewidth=2.5, marker="o", markersize=3, label="Train Accuracy")
axes[0].plot(text_lstm_history.history["val_accuracy"], color="#FD79A8", linewidth=2.5, marker="o", markersize=3, label="Validation Accuracy")
axes[0].set_title("Text LSTM — Accuracy Curve", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(text_lstm_history.history["loss"], color="#00B894", linewidth=2.5, marker="o", markersize=3, label="Train Loss")
axes[1].plot(text_lstm_history.history["val_loss"], color="#E17055", linewidth=2.5, marker="o", markersize=3, label="Validation Loss")
axes[1].set_title("Text LSTM — Loss Curve", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Text LSTM Training Performance — GoEmotions", fontsize=17, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

best_classical_text_acc = max(text_results.values())
print(f"\n🔍 Best Classical ML (TF-IDF) Accuracy : {best_classical_text_acc:.4f}")
print(f"🔍 LSTM (Embedding) Accuracy            : {lstm_text_acc:.4f}")

---
## STEP 32: TOP EMOTIONS CONFUSION MATRIX — COLORFUL CIRCULAR STYLE
---

In [ ]:
from sklearn.metrics import confusion_matrix

lstm_text_probs = text_lstm_model.predict(X_test_lstm)
lstm_text_preds = np.argmax(lstm_text_probs, axis=1)

top_10_emotions = go_df["emotion_label"].value_counts().head(10).index.tolist()
top_10_ids = [le_text.transform([e])[0] for e in top_10_emotions]

mask = np.isin(y_test_lstm, top_10_ids)
y_test_top10 = y_test_lstm[mask]
preds_top10 = lstm_text_preds[mask]

cm_top10 = confusion_matrix(y_test_top10, preds_top10, labels=top_10_ids)

fig, ax = plt.subplots(figsize=(11, 10))

cm_normalized = cm_top10 / cm_top10.sum(axis=1, keepdims=True)

im = ax.imshow(cm_normalized, cmap="magma", interpolation="nearest")

n = len(top_10_emotions)
for i in range(n):
    for j in range(n):
        value = cm_top10[i, j]
        intensity = cm_normalized[i, j]
        circle_size = 200 + (intensity * 2500)   # bigger circle = more confusion
        color = plt.cm.plasma(intensity)
        ax.scatter(j, i, s=circle_size, color=color, edgecolor="white", linewidth=1.5, alpha=0.9, zorder=3)
        if value > 0:
            ax.text(j, i, str(value), ha="center", va="center",
                     fontsize=9, fontweight="bold",
                     color="white" if intensity > 0.4 else "black", zorder=4)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(top_10_emotions, rotation=45, ha="right", fontsize=11, fontweight="bold")
ax.set_yticklabels(top_10_emotions, fontsize=11, fontweight="bold")
ax.set_xlabel("Predicted Emotion", fontsize=13, fontweight="bold")
ax.set_ylabel("True Emotion", fontsize=13, fontweight="bold")
ax.set_title("GoEmotions LSTM — Confusion Matrix (Top 10 Emotions)\nBubble size & color = confusion intensity",
              fontsize=15, fontweight="bold", pad=20)
ax.grid(alpha=0.2, zorder=0)
ax.set_facecolor("#f7f7f7")

plt.tight_layout()
plt.show()

cm_no_diag = cm_top10.copy()
np.fill_diagonal(cm_no_diag, 0)
max_idx = np.unravel_index(np.argmax(cm_no_diag), cm_no_diag.shape)
print(f"🔍 Most confused pair: True '{top_10_emotions[max_idx[0]]}' "
      f"predicted as '{top_10_emotions[max_idx[1]]}' ({cm_no_diag[max_idx]} times)")

---
## STEP 33: MULTIMODAL FUSION MODEL — FACE + VOICE + TEXT
---

In [ ]:
from tensorflow.keras import Input, Model

def build_feature_extractor(trained_model, input_shape):
    """
    Manually replays every layer of a trained model on a fresh Input tensor,
    EXCEPT the final output layer. This avoids relying on trained_model.input,
    which can fail to register in some Keras sessions.
    """
    new_input = Input(shape=input_shape)
    x = new_input
    # Pass through every layer except the very last one (final classification layer)
    for layer in trained_model.layers[:-1]:
        x = layer(x)
    return Model(inputs=new_input, outputs=x)

# ---- Rebuild all three feature extractors this way ----
face_feature_extractor = build_feature_extractor(cnn_model, input_shape=(48, 48, 1))
voice_feature_extractor = build_feature_extractor(lstm_audio, input_shape=(max_len, 40))
text_feature_extractor = build_feature_extractor(text_lstm_model, input_shape=(MAX_LEN,))

face_feature_extractor.trainable = False
voice_feature_extractor.trainable = False
text_feature_extractor.trainable = False

print("✅ Feature extractors rebuilt successfully.")
print(f"Face feature vector size : {face_feature_extractor.output_shape}")
print(f"Voice feature vector size: {voice_feature_extractor.output_shape}")
print(f"Text feature vector size : {text_feature_extractor.output_shape}")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Input, Model

COMMON_EMOTIONS = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Neutral"]

face_input  = Input(shape=(48, 48, 1), name="face_input")
voice_input = Input(shape=(max_len, 40), name="voice_input")
text_input  = Input(shape=(MAX_LEN,), name="text_input")

face_features  = face_feature_extractor(face_input)
voice_features = voice_feature_extractor(voice_input)
text_features  = text_feature_extractor(text_input)

fused_features = layers.Concatenate(name="fusion_layer")([face_features, voice_features, text_features])

x = layers.Dense(128, activation="relu")(fused_features)
x = layers.Dropout(0.4)(x)
x = layers.Dense(64, activation="relu")(x)
final_output = layers.Dense(len(COMMON_EMOTIONS), activation="softmax", name="final_emotion_output")(x)

fusion_model = Model(
    inputs=[face_input, voice_input, text_input],
    outputs=final_output,
    name="Multimodal_Emotion_Fusion_Model"
)

fusion_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fusion_model.summary()

print("\n✅ Fusion model built successfully!")

In [ ]:
try:
    tf.keras.utils.plot_model(
        fusion_model,
        to_file="fusion_model_architecture.png",
        show_shapes=True,
        show_layer_names=True,
        rankdir="TB",
        dpi=100
    )
    from IPython.display import Image, display
    display(Image(filename="fusion_model_architecture.png"))
except Exception as e:
    print(f"Diagram generation skipped (missing graphviz): {e}")
    print("Model summary above still shows the full architecture.")

---
## STEP 34: PREPARE MATCHED MULTIMODAL DATA + TRAIN FUSION MODEL
---

In [ ]:
COMMON_EMOTIONS = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Neutral"]

fer_mask = fer_df["emotion_label"].isin(COMMON_EMOTIONS)

ravdess_mask = ravdess_df["emotion"].isin(COMMON_EMOTIONS)

goemotions_map = {
    "anger": "Angry",
    "disgust": "Disgust",
    "fear": "Fear",
    "joy": "Happy",
    "sadness": "Sad",
    "neutral": "Neutral"
}
go_df["mapped_emotion"] = go_df["emotion_label"].map(goemotions_map)
go_mask = go_df["mapped_emotion"].notna()

print(f"FER2013 usable samples   : {fer_mask.sum()}")
print(f"RAVDESS usable samples   : {ravdess_mask.sum()}")
print(f"GoEmotions usable samples: {go_mask.sum()}")

SAMPLES_PER_CLASS = 150  

face_list, voice_list, text_list, label_list = [], [], [], []
rng = np.random.RandomState(SEED)

for emotion in COMMON_EMOTIONS:
   
    fer_idx = fer_df[fer_df["emotion_label"] == emotion].index
    rav_idx = ravdess_df[ravdess_df["emotion"] == emotion].index
    go_idx  = go_df[go_df["mapped_emotion"] == emotion].index

    n = min(len(fer_idx), len(rav_idx), len(go_idx), SAMPLES_PER_CLASS)
    if n == 0:
        print(f"⚠️ Skipping '{emotion}' — no overlap in one of the datasets")
        continue

    fer_sample = rng.choice(fer_idx, n, replace=False)
    rav_sample = rng.choice(rav_idx, n, replace=False)
    go_sample  = rng.choice(go_idx, n, replace=False)

    for i in range(n):
        # ---- Face: pixel array (already normalized in X_images_final, matched by fer_df row order) ----
        face_list.append(X_images_final[fer_sample[i]])

        # ---- Voice: MFCC sequence (extract fresh using the same function as Step 24) ----
        voice_list.append(get_mfcc_seq(ravdess_df.loc[rav_sample[i], "file_path"]))

        # ---- Text: padded token sequence (already computed in X_text_padded, matched by go_df row order) ----
        text_list.append(X_text_padded[go_sample[i]])

        label_list.append(COMMON_EMOTIONS.index(emotion))

X_face_fusion  = np.array(face_list)
X_voice_fusion = np.array(voice_list)
X_text_fusion  = np.array(text_list)
y_fusion       = np.array(label_list)

print(f"\n✅ Matched multimodal dataset created:")
print(f"   Face  : {X_face_fusion.shape}")
print(f"   Voice : {X_voice_fusion.shape}")
print(f"   Text  : {X_text_fusion.shape}")
print(f"   Labels: {y_fusion.shape}")

from sklearn.model_selection import train_test_split

indices = np.arange(len(y_fusion))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED, stratify=y_fusion)

X_face_train, X_face_test   = X_face_fusion[train_idx], X_face_fusion[test_idx]
X_voice_train, X_voice_test = X_voice_fusion[train_idx], X_voice_fusion[test_idx]
X_text_train, X_text_test   = X_text_fusion[train_idx], X_text_fusion[test_idx]
y_train_fusion, y_test_fusion = y_fusion[train_idx], y_fusion[test_idx]

fusion_history = fusion_model.fit(
    [X_face_train, X_voice_train, X_text_train], y_train_fusion,
    validation_split=0.15,
    epochs=20,
    batch_size=32,
    verbose=1
)

fusion_test_loss, fusion_test_acc = fusion_model.evaluate(
    [X_face_test, X_voice_test, X_text_test], y_test_fusion, verbose=0
)
print(f"\n✅ Fusion Model Test Accuracy: {fusion_test_acc:.4f}")

---
## STEP 35: FINAL COMPARISON — SINGLE MODALITY VS FUSION MODEL
---

In [ ]:
face_only_acc = cnn_model.evaluate(X_face_test, 
    np.array([COMMON_EMOTIONS.index(e) if isinstance(e, str) else e for e in 
              [list(fer_df["emotion_label"].unique())[0]]*len(y_test_fusion)]),  # placeholder, corrected below
    verbose=0)[1] if False else None

face_only_acc  = cnn_test_acc          # from Step 12/13, on FER2013's own test set
voice_only_acc = audio_results["LSTM"] # from Step 24, on RAVDESS's own test set
text_only_acc  = lstm_text_acc         # from Step 31, on GoEmotions' own test set

comparison_final = pd.DataFrame({
    "Model": ["Face Only\n(CNN)", "Voice Only\n(LSTM)", "Text Only\n(LSTM)", "Fusion Model\n(Face+Voice+Text)"],
    "Accuracy": [face_only_acc, voice_only_acc, text_only_acc, fusion_test_acc],
    "Type": ["Single Modality", "Single Modality", "Single Modality", "Multimodal Fusion"]
})

print("FINAL RESEARCH CONCLUSION — Modality Comparison:\n")
print(comparison_final.to_string(index=False))

colors = ["#74B9FF", "#A29BFE", "#FD79A8", "#00B894"]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.bar(comparison_final["Model"], comparison_final["Accuracy"],
              color=colors, edgecolor="white", linewidth=3, width=0.6)

for bar, acc in zip(bars, comparison_final["Accuracy"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
             f"{acc:.1%}", ha="center", fontsize=13, fontweight="bold")

if fusion_test_acc == comparison_final["Accuracy"].max():
    ax.text(3, fusion_test_acc + 0.08, "⭐ Best", ha="center", fontsize=14, fontweight="bold", color="#D63031")

ax.set_ylabel("Test Accuracy", fontsize=13, fontweight="bold")
ax.set_title("Final Research Conclusion:\nSingle Modality vs Multimodal Fusion",
              fontsize=16, fontweight="bold", pad=15)
ax.set_ylim(0, max(comparison_final["Accuracy"]) + 0.2)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

best_row = comparison_final.loc[comparison_final["Accuracy"].idxmax()]
print(f"\n📊 CONCLUSION: The best-performing approach was "
      f"'{best_row['Model'].replace(chr(10), ' ')}' with {best_row['Accuracy']:.1%} accuracy.")

if fusion_test_acc > max(face_only_acc, voice_only_acc, text_only_acc):
    improvement = (fusion_test_acc - max(face_only_acc, voice_only_acc, text_only_acc))
    print(f"✅ The Fusion Model outperformed every single modality by {improvement:.1%}, "
          f"confirming that combining Face + Voice + Text improves emotion recognition.")
else:
    print(f"⚠️ Note: The Fusion Model did not exceed the best single modality in this run. "
          f"This is an honest and common finding when modalities are pseudo-paired rather than "
          f"naturally co-recorded — discuss this as a limitation and future-work opportunity.")

=================================================================
---
## STEP 36A: CAPTURE LIVE PHOTO FROM WEBCAM (JavaScript + Python)
=================================================================
---


In [ ]:
try:
    if len(photo_upload.value) == 0 or len(voice_upload.value) == 0:
        print("⚠️ Please upload both a photo and a voice clip before running the next cell.")
        print("(This is expected if the notebook was auto-run without live uploads.)")
    else:
        # ... rest of your existing save code goes here
        pass
except NameError:
    print("⚠️ Upload widgets not yet created — please run the widget cell above first.")

In [ ]:
!pip install ipywidgets --quiet --break-system-packages

import ipywidgets as widgets
from IPython.display import display

# ---- Photo Upload Widget ----
photo_upload = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='📸 Upload Face Photo'
)

# ---- Voice Upload Widget ----
voice_upload = widgets.FileUpload(
    accept='audio/*',
    multiple=False,
    description='🎙️ Upload Voice Clip'
)

print("👉 Step 1: Take a photo of your face using your phone/laptop camera app, save it, then click below to upload it.")
display(photo_upload)

print("\n👉 Step 2: Record a short voice memo (e.g. using your phone's Voice Recorder app), save it, then click below to upload it.")
display(voice_upload)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import subprocess

def extract_uploaded_file(upload_widget):
    """Handles both old (dict) and new (tuple) ipywidgets FileUpload formats."""
    value = upload_widget.value
    if isinstance(value, dict):
        file_info = list(value.values())[0]
        content = file_info['content']
    else:
        file_info = value[0]
        content = file_info['content']
    return bytes(content)

# ---- Step 1: Save the uploaded photo to disk ----
if len(photo_upload.value) > 0:
    photo_bytes = extract_uploaded_file(photo_upload)
    with open("captured_face.jpg", "wb") as f:
        f.write(photo_bytes)
    print("✅ Photo saved as 'captured_face.jpg'")
else:
    print("⚠️ No photo uploaded yet — please upload before continuing.")

# ---- Step 2: Save the uploaded voice clip to disk ----
if len(voice_upload.value) > 0:
    voice_bytes = extract_uploaded_file(voice_upload)
    with open("recorded_voice_raw", "wb") as f:
        f.write(voice_bytes)
    print("✅ Voice file saved as 'recorded_voice_raw'")
else:
    print("⚠️ No voice file uploaded yet — please upload before continuing.")

# ---- Step 3: Convert voice to proper WAV format using ffmpeg ----
subprocess.run(["ffmpeg", "-y", "-i", "recorded_voice_raw", "recorded_voice.wav"],
               capture_output=True)
voice_wav_path = "recorded_voice.wav"
print("✅ Converted to 'recorded_voice.wav'")

# ---- Step 4: Text input widget ----
text_box = widgets.Textarea(
    value='',
    placeholder='Type a sentence describing how you feel... e.g. "I finally completed my project."',
    description='Your text:',
    layout=widgets.Layout(width='600px', height='60px')
)
display(text_box)

print("\n👉 Type your sentence in the box above. Once done, run the NEXT cell to see your final emotion prediction!")

---
## STEP 37: FINAL LIVE PREDICTION — FACE + VOICE + TEXT -> EMOTION
---

In [ ]:
import numpy as np
import librosa

from PIL import Image
import matplotlib.pyplot as plt

COMMON_EMOTIONS = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Neutral"]

# ---- Step 1: Preprocess the FACE photo (same steps as FER2013 training) ----
face_img = Image.open("captured_face.jpg").convert("L")   # convert to grayscale
face_img = face_img.resize((48, 48))                       # match training size
face_array = np.array(face_img, dtype="float32") / 255.0   # normalize 0-1
face_input_final = face_array.reshape(1, 48, 48, 1)        # add batch + channel dims

# ---- Step 2: Preprocess the VOICE recording (same steps as RAVDESS training) ----
def get_mfcc_seq_live(path, max_len=130, n_mfcc=40):
    y_audio, sr = librosa.load(path, sr=22050)
    mfcc = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=n_mfcc).T
    if mfcc.shape[0] < max_len:
        mfcc = np.pad(mfcc, ((0, max_len - mfcc.shape[0]), (0, 0)))
    else:
        mfcc = mfcc[:max_len]
    return mfcc

voice_features_live = get_mfcc_seq_live(voice_wav_path)
voice_input_final = voice_features_live.reshape(1, 130, 40)

# ---- Step 3: Preprocess the TEXT sentence (same steps as GoEmotions training) ----
user_sentence = text_box.value
cleaned_sentence = clean_text_safe(user_sentence)   # reuse Step 27's cleaning function
sequence = tokenizer.texts_to_sequences([cleaned_sentence])
text_input_final = pad_sequences(sequence, maxlen=MAX_LEN, padding="post", truncating="post")

print(f"📸 Face image  : preprocessed to {face_input_final.shape}")
print(f"🎙️ Voice clip  : preprocessed to {voice_input_final.shape}")
print(f"📝 Text input  : \"{user_sentence}\" -> cleaned: \"{cleaned_sentence}\"")

# ---- Step 4: Feed all three into the Fusion Model ----
fusion_prediction = fusion_model.predict([face_input_final, voice_input_final, text_input_final], verbose=0)
predicted_class = np.argmax(fusion_prediction, axis=1)[0]
predicted_emotion = COMMON_EMOTIONS[predicted_class]
confidence = fusion_prediction[0][predicted_class]

# ---- Step 5: Display everything beautifully ----
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(face_array, cmap="gray")
axes[0].set_title("Your Face", fontsize=13, fontweight="bold")
axes[0].axis("off")

axes[1].bar(COMMON_EMOTIONS, fusion_prediction[0], color=plt.cm.plasma(np.linspace(0.2, 0.9, len(COMMON_EMOTIONS))))
axes[1].set_title("Model Confidence per Emotion", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Probability")
axes[1].tick_params(axis='x', rotation=30)

axes[2].axis("off")
axes[2].text(0.5, 0.6, "⭐⭐ FINAL PREDICTION ⭐⭐", ha="center", fontsize=15, fontweight="bold", color="#D63031")
axes[2].text(0.5, 0.35, predicted_emotion.upper(), ha="center", fontsize=32, fontweight="bold", color="#00B894")
axes[2].text(0.5, 0.15, f"Confidence: {confidence:.1%}", ha="center", fontsize=13, color="gray")

plt.suptitle("Multimodal Emotion AI — Live Prediction Result", fontsize=17, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

print(f"\n🎉 Final Predicted Emotion: {predicted_emotion} (Confidence: {confidence:.1%})")

# ===================================================
# 🎯 Final Project Summary — Multimodal Emotion AI
# ===================================================
## Project Pipeline Overview
This project built and compared emotion recognition systems across three modalities — 
Facial Expressions (FER2013), Speech (RAVDESS), and Text (GoEmotions) — before combining 
them into a single Multimodal Fusion Model.

## Key Findings

| Module | Best Model | Key Insight |
|---|---|---|
| Facial (FER2013) | CNN | Spatial feature learning outperformed classical ML and even LSTM |
| Speech (RAVDESS) | LSTM | Sequential modeling suited audio's time-series nature |
| Text (GoEmotions) | LSTM + Embeddings | Word order/context mattered more than bag-of-words (TF-IDF) |
| Fusion | Multimodal Model | Combined Face+Voice+Text into one final prediction |

## Techniques Demonstrated
- **Statistics:** class distributions, skewness/kurtosis, class weight balancing
- **Data Visualization:** 15+ chart types (histograms, 3D surfaces, word clouds, spectrograms, etc.)
- **Machine Learning:** Logistic Regression, Decision Tree, Random Forest, SVM, Naive Bayes
- **Deep Learning:** ANN, CNN, LSTM
- **Transfer Learning:** MobileNetV2, EfficientNetB0
- **NLP:** Tokenization, embeddings, text cleaning, TF-IDF
- **Speech Processing:** MFCC, Spectrograms, Librosa
- **Live Deployment:** Real-time camera + microphone + text input demo

## Honest Limitations (Important for Q&A)
- The three datasets were not naturally co-recorded — fusion training used **pseudo-paired** 
  samples matched by emotion label, a standard technique when synchronized multimodal data isn't available.
- GoEmotions' 28 fine-grained emotions were mapped down to 6 shared categories for fusion compatibility.
- Future work: a custom-recorded, truly synchronized multimodal dataset would likely improve fusion performance further.

---
## STEP 38: AUTO-GENERATED FINAL METRICS TABLE (For Presentation)
---

In [ ]:
final_report = pd.DataFrame({
    "Module": ["Facial (FER2013)", "Facial (FER2013)", "Facial (FER2013)", "Facial (FER2013)",
               "Speech (RAVDESS)", "Text (GoEmotions)", "Multimodal Fusion"],
    "Model": ["Classical ML (best)", "CNN", "MobileNetV2", "EfficientNetB0",
              "LSTM", "LSTM", "Fusion (Face+Voice+Text)"],
    "Test Accuracy": [best_classical_acc, cnn_test_acc, mnv2_test_acc, effnet_test_acc,
                       audio_results["LSTM"], lstm_text_acc, fusion_test_acc]
})

final_report["Test Accuracy"] = final_report["Test Accuracy"].apply(lambda x: f"{x:.1%}")
print("📋 FINAL PROJECT METRICS TABLE (copy this into your report/slides):\n")
display(final_report)

final_report.to_csv("final_project_metrics.csv", index=False)
print("\n✅ Saved as 'final_project_metrics.csv'.")

---
##  **Step 39: Complete Fusion Model Evaluation Report**
---

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

COMMON_EMOTIONS = ["Angry", "Disgust", "Fear", "Happy", "Sad", "Neutral"]

# ---- Step 1: Get predictions from the Fusion Model on the test set ----
fusion_probs = fusion_model.predict([X_face_test, X_voice_test, X_text_test], verbose=0)
fusion_preds = np.argmax(fusion_probs, axis=1)

# ---- Step 2: Overall metrics ----
overall_accuracy  = accuracy_score(y_test_fusion, fusion_preds)
overall_precision = precision_score(y_test_fusion, fusion_preds, average="weighted", zero_division=0)
overall_recall    = recall_score(y_test_fusion, fusion_preds, average="weighted", zero_division=0)
overall_f1        = f1_score(y_test_fusion, fusion_preds, average="weighted", zero_division=0)

print("=" * 55)
print("       FUSION MODEL — OVERALL PERFORMANCE")
print("=" * 55)
print(f"  Accuracy   : {overall_accuracy:.4f}  ({overall_accuracy:.1%})")
print(f"  Precision  : {overall_precision:.4f}")
print(f"  Recall     : {overall_recall:.4f}")
print(f"  F1-Score   : {overall_f1:.4f}")
print(f"  Total Test Samples: {len(y_test_fusion)}")
print("=" * 55)

# ---- Step 3: Per-class detailed report ----
print("\nPer-Class Detailed Report:\n")
report = classification_report(y_test_fusion, fusion_preds, target_names=COMMON_EMOTIONS, zero_division=0)
print(report)

# ---- Step 4: Confusion Matrix ----
cm = confusion_matrix(y_test_fusion, fusion_preds)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap version
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=COMMON_EMOTIONS, yticklabels=COMMON_EMOTIONS, ax=axes[0])
axes[0].set_title("Fusion Model — Confusion Matrix", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Predicted Emotion")
axes[0].set_ylabel("True Emotion")

# Per-class accuracy bar chart
per_class_acc = cm.diagonal() / cm.sum(axis=1)
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(COMMON_EMOTIONS)))
bars = axes[1].bar(COMMON_EMOTIONS, per_class_acc, color=colors, edgecolor="white", linewidth=2)
for bar, acc in zip(bars, per_class_acc):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{acc:.1%}", ha="center", fontsize=10, fontweight="bold")
axes[1].set_title("Fusion Model — Per-Class Accuracy", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.15)
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle("Fusion Model — Complete Evaluation Report", fontsize=17, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

# ---- Step 5: Summary table (good for report/slides) ----
eval_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (weighted)", "Recall (weighted)", "F1-Score (weighted)"],
    "Value": [overall_accuracy, overall_precision, overall_recall, overall_f1]
})
eval_summary["Value"] = eval_summary["Value"].apply(lambda x: f"{x:.1%}")
display(eval_summary)

eval_summary.to_csv("fusion_model_evaluation_summary.csv", index=False)
print("\n✅ Saved as 'fusion_model_evaluation_summary.csv'")

In [ ]:
import keras
from keras import layers, Model

# 1. Text Branch Definition
text_input = layers.Input(shape=(30,), name="text_input")
x_text = layers.Embedding(input_dim=10000, output_dim=100)(text_input)
x_text = layers.LSTM(128, return_sequences=True)(x_text)
x_text = layers.Dropout(0.3)(x_text)
x_text = layers.LSTM(64)(x_text)
x_text = layers.Dense(64, activation="relu")(x_text)

# 2. Audio Branch Definition (Example Shape: adjust as per your data)
audio_input = layers.Input(shape=(128, 1), name="audio_input") 
x_audio = layers.Conv1D(64, kernel_size=3, activation="relu")(audio_input)
x_audio = layers.GlobalAveragePooling1D()(x_audio)
x_audio = layers.Dense(64, activation="relu")(x_audio)

# 3. Image Branch Definition (Example Shape: adjust as per your data)
image_input = layers.Input(shape=(224, 224, 3), name="image_input")
x_image = layers.Conv2D(32, (3, 3), activation="relu")(image_input)
x_image = layers.GlobalAveragePooling2D()(x_image)
x_image = layers.Dense(64, activation="relu")(x_image)

# 4. Fusion Layer (Teeno Branches ko Milana)
fused = layers.concatenate([x_text, x_audio, x_image])
x = layers.Dense(128, activation="relu")(fused)
x = layers.Dropout(0.3)(x)
output = layers.Dense(28, activation="softmax", name="emotion_output")(x)

# 5. Full Multimodal Model Construct Karein
full_multimodal_model = Model(
    inputs=[text_input, audio_input, image_input], 
    outputs=output, 
    name="Multimodal_Emotion_Model"
)

# NOTE: Agar aap ne pehle se model train kiya hua hai, toh uss 'model' object ko direct save karein:
# Full Model Save Karein Native Keras Format Mein
full_multimodal_model.save("full_multimodal_emotion_model.keras")

print("Full Multimodal Model Successfully Saved!")

In [ ]:
import keras

# Load Full Multimodal Model
model = keras.models.load_model("full_multimodal_emotion_model.keras")

# Verification
model.summary()  # Isme 3 inputs aur concatenate layer nazar aani chahiye

In [ ]:
text_lstm_model.save('emotion_recognition_model.h5')
print("Model weights successfully saved!")
text_lstm_model.save('emotion_recognition_model.keras')